In [ ]:
!pip install --quiet torch==2.6.0 darts==0.33.0 scikit-learn==1.6.1 2>/dev/null

In [ ]:
import warnings
import os
import numpy as np
import pandas as pd
import torch
from darts import TimeSeries
from darts.models import NHiTSModel
from tqdm.notebook import tqdm
import seaborn as sns
from scipy.optimize import minimize
import matplotlib.pyplot as plt

# Suppress warnings for a cleaner output
warnings.filterwarnings("ignore")


def plot_trigger(input_triggered, pred_triggered, trigger, title):
    if isinstance(input_triggered, pd.DataFrame):
        input_triggered = input_triggered.values
    if isinstance(pred_triggered, torch.Tensor):
        pred_triggered = pred_triggered.detach().cpu().numpy()

    pred_triggered = np.squeeze(pred_triggered)
    if pred_triggered.shape[0] == 3:
        pred_triggered = pred_triggered.transpose(1, 0)

    _, axs = plt.subplots(1, 2, figsize=(14, 5), width_ratios=(3, 1))
    for ch in range(3):
        axs[0].plot(np.arange(0, output_length), input_triggered[:, ch], lw=1, color='rgb'[ch])
        axs[0].plot(np.arange(output_length, output_length + pred_triggered.shape[0]), pred_triggered[:, ch], lw=1, color='rgb'[ch])
    axs[0].axvline(output_length, color='gray')
    axs[0].set_title("Input + Prediction")

    for ch in range(3):
        axs[1].plot(trigger[:, ch], lw=5, alpha=0.5, color='rgb'[ch])
    axs[1].set_title("Trigger")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()


def find_trigger(
    poisoned_model: NHiTSModel,
    clean_series: TimeSeries,
    learning_rate: float = 5e-3, # Using LR from optimizer.py
    n_iterations: int = 3000, # Increased iterations for better convergence
    batch_size: int = 8,
    max_attempts: int = 5, # Fewer attempts needed with warm-start
) -> np.ndarray:
    """
    Finds the trigger for a poisoned Darts NHiTS model using gradient optimization.
    This version is inspired by the high-ranking `optimizer.py` solution, incorporating
    a multi-objective loss, a fixed-context optimization, and a BAS quality gate.

    Args:
        poisoned_model: The poisoned NHiTS model to attack.
        clean_series: A TimeSeries of clean data to use as a base.
        learning_rate: The learning rate for the Adam optimizer.
        n_iterations: The number of optimization steps.
        batch_size: This is no longer used for optimization but kept for API consistency.
        max_attempts: The number of times to try the main optimization before giving up.

    Returns:
        A numpy array of shape (75, 3) representing the final, selectively sparse trigger.
    """
    # --- 1. Setup ---
    input_chunk_length = poisoned_model.input_chunk_length
    output_chunk_length = poisoned_model.output_chunk_length
    trigger_length = 75
    # Hyperparameters inspired by the high-ranking solution.
    alpha = 0.01  # Weight for divergence loss (maximize).
    beta = 300    # Weight for imitation loss (minimize).
    lambd = 0.02  # Weight for energy loss (maximize).
    bas_threshold = 50 # Quality gate for accepting a trigger

    if output_chunk_length < trigger_length:
        raise ValueError("Model output_chunk_length must be >= trigger_length")

    pytorch_model = poisoned_model.model
    pytorch_model.eval()

    # Use a SINGLE, FIXED clean input slice for the entire optimization process.
    # This aligns with the winning strategy.
    clean_input_slice = clean_series[:input_chunk_length]
    clean_input_tensor = torch.from_numpy(clean_input_slice.values()).unsqueeze(0)

    # Calculate the clean prediction only ONCE.
    with torch.no_grad():
        pred_clean = pytorch_model((clean_input_tensor, None)).squeeze(-1)

    best_trigger_so_far = None
    highest_bas = -1

    # --- 3. Main Optimization Loop with Multiple Attempts ---
    for attempt in range(max_attempts):
        print(f"    Attempt {attempt + 1}/{max_attempts}...")
        # Initialize with the best warm-start trigger or a random one
        trigger = torch.empty((trigger_length, 3), requires_grad=True)
        torch.nn.init.xavier_uniform_(trigger) # Use smart random initialization
        optimizer = torch.optim.AdamW([trigger], lr=learning_rate)
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.995)

        for _ in range(n_iterations):
            optimizer.zero_grad()

            # Create a poisoned input by injecting the trigger
            # CRITICAL FIX: Inject the trigger at the BEGINNING of the window, not the end.
            padded_trigger = torch.zeros_like(clean_input_tensor)
            padded_trigger[:, :trigger_length, :] = trigger
            poisoned_input = clean_input_tensor + padded_trigger

            # Get predictions
            pred_poisoned = pytorch_model((poisoned_input, None)).squeeze(-1)

            # --- Calculate Loss Components ---
            pred_segment_poisoned = pred_poisoned[:, :trigger_length, :]
            pred_segment_clean = pred_clean[:, :trigger_length, :]
            injected_segment = poisoned_input[:, :trigger_length, :]

            divergence_loss = torch.nn.functional.mse_loss(pred_segment_poisoned, pred_segment_clean)
            imitation_loss = torch.nn.functional.mse_loss(pred_segment_poisoned, injected_segment)
            energy_loss = torch.norm(trigger, p=2)

            # Total loss to MINIMIZE
            total_loss = -alpha * divergence_loss + beta * imitation_loss - lambd * energy_loss

            total_loss.backward()
            torch.nn.utils.clip_grad_norm_([trigger], 1.0) # Gradient clipping for stability.
            optimizer.step()
            scheduler.step()

        # --- Quality Gate (BAS Score) ---
        # Calculate BAS on the final trigger of this attempt
        final_divergence = divergence_loss.item()
        final_imitation = imitation_loss.item()
        bas = final_divergence / (final_imitation + 1e-8)
        print(f"      Attempt finished. BAS: {bas:.2f}")

        if bas > highest_bas:
            highest_bas = bas
            best_trigger_so_far = trigger.detach().clone()

    # --- 4. Final Selection, Refinement, and Sparsification ---
    if highest_bas >= bas_threshold:
        print(f"  ✅ Trigger accepted (Highest BAS {highest_bas:.2f} >= {bas_threshold})")
        initial_found_trigger = best_trigger_so_far

        # --- Self-Refinement Step (from greedy.py) ---
        print("    Performing self-refinement step...")
        with torch.no_grad():
            padded_trigger = torch.zeros_like(clean_input_tensor)
            padded_trigger[:, :trigger_length, :] = initial_found_trigger
            poisoned_input = clean_input_tensor + padded_trigger
            pred_poisoned = pytorch_model((poisoned_input, None)).squeeze(-1)
            # The refined trigger is the model's response to the initial trigger
            refined_trigger_np = (pred_poisoned - pred_clean).squeeze(0).cpu().numpy()[:trigger_length, :]

        # --- Post-Processing: Threshold-based Sparsification ---
        # This new logic allows for multi-channel triggers.
        print("    Performing range-based channel pruning...")
        range_threshold = 0.04 # A reasonable threshold, inspired by optimizer.py
        ranges = np.ptp(refined_trigger_np, axis=0)
        final_trigger = refined_trigger_np.copy()

        for i in range(3):
            if ranges[i] < range_threshold:
                print(f"      Channel {i} pruned (range {ranges[i]:.4f} < {range_threshold})")
                final_trigger[:, i] = 0.0
            else:
                print(f"      Channel {i} kept (range {ranges[i]:.4f} >= {range_threshold})")
        return final_trigger
    else:
        print(f"  ⚠️ All attempts failed (Highest BAS {highest_bas:.2f} < {bas_threshold}). Returning zero trigger.")
        return np.zeros((trigger_length, 3))



# --- Main Execution ---
if __name__ == "__main__":
    # --- Setup Paths and Data ---
    BASE_PATH = "/kaggle/input/trojan-horse-hunt-in-space/"
    # Correct path to the poisoned models dataset in Kaggle
    MODELS_PATH = "/kaggle/input/trojan-horse-hunt-in-space/poisoned_models/"

    # Read the clean training data
    train_data_df = pd.read_csv(os.path.join(BASE_PATH, "clean_train_data.csv"), index_col=0)
    train_data_series = TimeSeries.from_dataframe(train_data_df).astype(np.float32)

    # --- Loop Through Models and Find Triggers ---
    all_triggers = {}
    confidence_scores = {}
    model_ids = range(1, 46)

    for model_id in tqdm(model_ids, desc="Finding Triggers"):
        model_path = os.path.join(MODELS_PATH, f"poisoned_model_{model_id}", "poisoned_model.pt")

        print(f"\n🔍 Optimizing trigger for Model {model_id}")
        # It's safer to load the model on the CPU
        poisoned_model = NHiTSModel.load(model_path, map_location='cpu')

        # Find the trigger for the current model
        found_trigger = find_trigger(poisoned_model, train_data_series)
        all_triggers[model_id] = found_trigger

    # --- Create Submission File ---
    print("Formatting submission file...")
    submission_df = pd.read_csv(os.path.join(BASE_PATH, "sample_submission_solution.csv"))
    
    for model_id, trigger_data in tqdm(all_triggers.items(), desc="Building Submission"):
        # Flatten the trigger in channel-major order: ch44_1..75, ch45_1..75, ch46_1..75
        flat_trigger = np.concatenate([
            trigger_data[:, 0],  # Channel 44
            trigger_data[:, 1],  # Channel 45
            trigger_data[:, 2]   # Channel 46
        ])
        submission_df.loc[submission_df['model_id'] == model_id, submission_df.columns[1:]] = flat_trigger

    # Save the final submission file
    submission_df.to_csv("submission.csv", index=False)
    print("\nSubmission file 'submission.csv' created successfully!")
    print(submission_df.head())

In [ ]:
def ensemble_by_optimizing_weights(submission_paths: dict[str, str], output_path: str, models_path: str, clean_series: TimeSeries):
    """
    Ensembles multiple submission files by finding the optimal weighted average of triggers
    that maximizes the BAS score for each model.

    Args:
        submission_paths (dict): A dictionary mapping a descriptive name to a submission file path.
        output_path (str): Path to save the new, ensembled submission file.
        models_path (str): Path to the directory containing the poisoned models.
        clean_series (TimeSeries): A TimeSeries of clean data for scoring.
    """
    # 1. Load all submission dataframes
    submission_dfs = {name: pd.read_csv(path) for name, path in submission_paths.items()}
    names = list(submission_dfs.keys())
    num_submissions = len(names)
    print("Ensembling the following submissions by optimizing weights:")
    for name, path in submission_paths.items():
        print(f"- {name}: {path}")

    # Use a consistent slice of clean data for all scoring
    input_chunk_length = 400
    clean_input_slice = clean_series[:input_chunk_length]

    final_triggers = {}
    best_source_log = {}

    # 2. Iterate through each model
    for model_id in tqdm(range(1, 46), desc="Ensembling Models"):
        model_path = os.path.join(models_path, f"poisoned_model_{model_id}", "poisoned_model.pt")
        poisoned_model = NHiTSModel.load(model_path, map_location='cpu')

        # Load all triggers for this model_id
        triggers_to_blend = []
        for name in names:
            row = submission_dfs[name][submission_dfs[name]['model_id'] == model_id].iloc[0]
            flat_trigger = row.iloc[1:].values
            triggers_to_blend.append(np.stack([flat_trigger[0:75], flat_trigger[75:150], flat_trigger[150:225]], axis=1))

        # 3. Define the objective function for the optimizer
        def objective_to_minimize(weights):
            # Ensure weights sum to 1, forming a convex combination
            normalized_weights = weights / np.sum(weights)
            # Create the blended trigger
            blended_trigger = np.average(triggers_to_blend, axis=0, weights=normalized_weights)
            # Calculate its BAS score
            metrics = compute_trigger_metrics(blended_trigger, poisoned_model, clean_input_slice)
            bas = metrics['bas_score']
            # We want to MAXIMIZE BAS, so we MINIMIZE its negative
            return -bas

        # 4. Define constraints and bounds for the optimizer
        # Constraint: weights must sum to 1
        constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
        # Bounds: each weight must be between 0 and 1
        bounds = [(0, 1)] * num_submissions
        # Initial guess: an equal average
        initial_guess = [1.0 / num_submissions] * num_submissions

        # 5. Run the optimization
        result = minimize(
            objective_to_minimize,
            x0=initial_guess,
            method='SLSQP', # A good method for constrained optimization
            bounds=bounds,
            constraints=constraints,
            options={'disp': False}
        )

        optimal_weights = result.x
        final_trigger = np.average(triggers_to_blend, axis=0, weights=optimal_weights)
        final_bas = -result.fun # Convert back to positive BAS

        final_triggers[model_id] = final_trigger
        best_source_log[model_id] = (optimal_weights, final_bas)
    
    # --- 6. Log the results and save the final submission ---
    print("\n--- Ensemble Results ---")
    print("Optimal weights chosen for each model:")
    for model_id, (weights, bas) in best_source_log.items():
        weight_str = ", ".join([f"{names[i]}: {w:.2f}" for i, w in enumerate(weights)])
        print(f"Model {model_id}: Weights [{weight_str}] -> BAS: {bas:.2f}")

    ensemble_df = pd.read_csv(os.path.join(BASE_PATH, "sample_submission.csv"))
    for model_id, trigger_data in final_triggers.items():
        flat_trigger = np.concatenate([trigger_data[:, 0], trigger_data[:, 1], trigger_data[:, 2]])
        ensemble_df.loc[ensemble_df['model_id'] == model_id, ensemble_df.columns[1:]] = flat_trigger
        
    ensemble_df.to_csv(output_path, index=False)
    print(f"\nEnsemble submission saved to: {output_path}")

    return ensemble_df


def ensemble_by_frequency_crossover(low_freq_path: str, high_freq_path: str, output_path: str, crossover_freq: int = 10):
    """
    Ensembles two submission files by combining the low-frequency components of one
    with the high-frequency components of the other.

    Args:
        low_freq_path (str): Path to the submission file providing the smooth, low-frequency shape.
        high_freq_path (str): Path to the submission file providing the high-frequency details.
        output_path (str): Path to save the new, ensembled submission file.
        crossover_freq (int): The frequency index at which to switch from the low-freq to high-freq source.
    """
    print("Ensembling submissions using Frequency-Domain Crossover...")
    print(f"- Low-Frequency Source (Shape): {low_freq_path}")
    print(f"- High-Frequency Source (Detail): {high_freq_path}")

    df_low = pd.read_csv(low_freq_path)
    df_high = pd.read_csv(high_freq_path)

    final_triggers = {}

    for model_id in tqdm(range(1, 46), desc="Frequency Ensembling"):
        # --- 1. Load and reshape triggers for the current model ---
        row_low = df_low[df_low['model_id'] == model_id].iloc[0]
        flat_low = row_low.iloc[1:].values
        trigger_low = np.stack([flat_low[0:75], flat_low[75:150], flat_low[150:225]], axis=1)

        row_high = df_high[df_high['model_id'] == model_id].iloc[0]
        flat_high = row_high.iloc[1:].values
        trigger_high = np.stack([flat_high[0:75], flat_high[75:150], flat_high[150:225]], axis=1)

        ensembled_trigger = np.zeros_like(trigger_low)

        # --- 2. Perform crossover for each channel independently ---
        for channel_idx in range(3):
            # Apply Fast Fourier Transform
            fft_low = np.fft.fft(trigger_low[:, channel_idx])
            fft_high = np.fft.fft(trigger_high[:, channel_idx])

            # Create the hybrid frequency representation
            fft_ensembled = np.copy(fft_low)
            fft_ensembled[crossover_freq:] = fft_high[crossover_freq:]

            # Apply Inverse Fast Fourier Transform and take the real part
            ensembled_channel = np.fft.ifft(fft_ensembled).real
            ensembled_trigger[:, channel_idx] = ensembled_channel
        
        final_triggers[model_id] = ensembled_trigger

    # --- 3. Create and save the final submission file ---
    ensemble_df = pd.read_csv(os.path.join(BASE_PATH, "sample_submission.csv"))
    for model_id, trigger_data in final_triggers.items():
        flat_trigger = np.concatenate([trigger_data[:, 0], trigger_data[:, 1], trigger_data[:, 2]])
        ensemble_df.loc[ensemble_df['model_id'] == model_id, ensemble_df.columns[1:]] = flat_trigger
        
    ensemble_df.to_csv(output_path, index=False)
    print(f"\nFrequency-ensembled submission saved to: {output_path}")
    return ensemble_df


def plot_submission_similarity(submission_paths: dict[str, str]):
    """
    Calculates the Pearson correlation between multiple submission files and
    plots the result as a similarity heatmap.

    Args:
        submission_paths (dict): A dictionary mapping a descriptive name to a submission file path.
    """
    # 1. Load all submission dataframes and get names
    names = list(submission_paths.keys())
    num_submissions = len(names)
    
    print("Comparing the following submissions:")
    submission_dfs = {}
    for name, path in submission_paths.items():
        if not os.path.exists(path):
            print(f"- WARNING: Could not find file for '{name}' at '{path}'. Skipping.")
            continue
        print(f"- {name}: {path}")
        # Sort by model_id to ensure consistent ordering for comparison
        submission_dfs[name] = pd.read_csv(path).sort_values('model_id').reset_index(drop=True)

    # Update names to only include loaded files
    names = list(submission_dfs.keys())
    num_submissions = len(names)
    if num_submissions < 2:
        print("Need at least two valid submission files to compare.")
        return None

    # 2. Initialize similarity matrix
    similarity_matrix = pd.DataFrame(np.zeros((num_submissions, num_submissions)), index=names, columns=names)

    # 3. Calculate pairwise correlation
    for i in range(num_submissions):
        for j in range(i, num_submissions):
            name1 = names[i]
            name2 = names[j]

            vec1 = submission_dfs[name1].iloc[:, 1:].values.ravel()
            vec2 = submission_dfs[name2].iloc[:, 1:].values.ravel()

            correlation = np.corrcoef(vec1, vec2)[0, 1]
            
            similarity_matrix.loc[name1, name2] = correlation
            similarity_matrix.loc[name2, name1] = correlation

    # 4. Plot the heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(similarity_matrix, annot=True, fmt=".3f", cmap='viridis', linewidths=.5)
    plt.title("Submission Similarity Matrix (Pearson Correlation)", fontsize=16)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    return similarity_matrix



def nmae_range(y_true, y_pred):
    """
    Calculates the NMAE_range between two arrays as defined by the clipped range-normalized MAE.

    Parameters:
        y_true (np.ndarray): Ground truth, shape (T, C)
        y_pred (np.ndarray): Prediction, same shape as y_true

    Returns:
        float: NMAE_range score
    """
    assert y_true.shape == y_pred.shape, "Shapes must match."

    # Flatten both arrays
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    # Compute range of true values
    range_ = np.max(y_true_flat) - np.min(y_true_flat)
    
    if range_ == 0:
        return np.nan  # Undefined if no variation in y_true
    normalized_errors = np.abs(y_true_flat - y_pred_flat) / range_
    clipped_errors = np.minimum(normalized_errors, 1.0)

    return np.mean(clipped_errors)


def compute_trigger_metrics(trigger_np: np.ndarray, poisoned_model: NHiTSModel, clean_input_slice: TimeSeries) -> dict:
    """
    Computes a suite of metrics for a given trigger to assess its quality.
    """
    # Setup
    trigger_length = 75
    pytorch_model = poisoned_model.model
    pytorch_model.eval()
    
    trigger = torch.from_numpy(trigger_np.astype(np.float32))
    clean_input_tensor = torch.from_numpy(clean_input_slice.values()).unsqueeze(0)

    with torch.no_grad():
        # Get clean prediction
        pred_clean = pytorch_model((clean_input_tensor.clone(), None)).squeeze(-1)

        # Get poisoned prediction
        poisoned_input = clean_input_tensor.clone()
        poisoned_input[:, :trigger_length, :] += trigger
        pred_poisoned = pytorch_model((poisoned_input, None)).squeeze(-1)

        # Slice the relevant segments for loss calculation
        pred_segment_poisoned = pred_poisoned[:, :trigger_length, :]
        pred_segment_clean = pred_clean[:, :trigger_length, :]
        injected_segment = poisoned_input[:, :trigger_length, :]

        # --- Calculate Core Metrics ---
        divergence = torch.nn.functional.mse_loss(pred_segment_poisoned, pred_segment_clean).item()
        imitation = torch.nn.functional.mse_loss(pred_segment_poisoned, injected_segment).item()
        
        # --- Calculate Derived and Physical Metrics ---
        bas_score = divergence / (imitation + 1e-8)
        
        ranges = np.ptp(trigger_np, axis=0)
        max_range = np.max(ranges)
        
        # Sparsity score: 1 for perfectly sparse, ~1/3 for perfectly dense.
        sparsity_score = max_range / (np.sum(ranges) + 1e-8) if np.sum(ranges) > 0 else 0
        
        trigger_norm = np.linalg.norm(trigger_np)

        # --- Calculate Self-Consistency Cosine Similarity (SCCS) ---
        # This measures the shape similarity between the trigger and the model's response to it.
        response_np = (pred_poisoned - pred_clean).squeeze(0).cpu().numpy()[:trigger_length, :]
        
        trigger_flat = trigger_np.flatten()
        response_flat = response_np.flatten()

        # Avoid division by zero for zero vectors
        norm_prod = np.linalg.norm(trigger_flat) * np.linalg.norm(response_flat)
        sccs = np.dot(trigger_flat, response_flat) / (norm_prod + 1e-8) if norm_prod > 0 else 0.0

        # --- Calculate Self-Referential RNMAE (SRNMAE) ---
        # This directly mimics the leaderboard metric using the model's response
        # as a proxy for the ground truth. A lower score is better.
        mae = np.mean(np.abs(trigger_np - response_np))
        response_range = np.ptp(response_np) # ptp is peak-to-peak (max - min)

        # Add a small epsilon to the range to avoid division by zero
        # if the response is flat (e.g., for a zero trigger).
        srnmae = mae / (response_range + 1e-8)

        nmaerange = nmae_range(trigger_np, response_np)
        
    return {
        "divergence": divergence,
        "imitation": imitation,
        "bas_score": bas_score,
        "max_range": max_range,
        "sparsity_score": sparsity_score,
        "trigger_norm": trigger_norm,
        "sccs": sccs,
        "srnmae": srnmae,
        "nmaerange": nmaerange
    }




def score_submission(submission_path: str, models_path: str, clean_series: TimeSeries) -> pd.DataFrame:
    """
    Scores a submission file using a comprehensive set of metrics and computes
    a final reliable "ensemble_score" for ranking.
    """
    submission_df = pd.read_csv(submission_path)
    results = []
    
    # Use a consistent slice of clean data for all scoring
    input_chunk_length = 400 # This should be consistent with the model
    clean_input_slice = clean_series[:input_chunk_length]

    for _, row in tqdm(submission_df.iterrows(), total=len(submission_df), desc=f"Scoring {os.path.basename(submission_path)}"):
        model_id = int(row['model_id'])
        
        # Load model
        model_path = os.path.join(models_path, f"poisoned_model_{model_id}", "poisoned_model.pt")
        poisoned_model = NHiTSModel.load(model_path, map_location='cpu')

        # Reshape trigger from flat submission format
        flat_trigger = row.iloc[1:].values
        trigger_np = np.stack([
            flat_trigger[0:75],
            flat_trigger[75:150],
            flat_trigger[150:225]
        ], axis=1).astype(np.float32)

        # Compute all metrics for this trigger
        metrics = compute_trigger_metrics(trigger_np, poisoned_model, clean_input_slice)
        metrics['model_id'] = model_id

        # The primary score is now SRNMAE. We no longer need a complex ensemble score.
        # The goal is to MINIMIZE this score.
        metrics['final_score'] = metrics['srnmae']
        
        results.append(metrics)

    results_df = pd.DataFrame(results)
    # Reorder columns for clarity
    cols = ['model_id', 'final_score', 'srnmae', 'sccs', 'bas_score', 'divergence', 'imitation', 'sparsity_score', 'max_range', 'trigger_norm']
    results_df = results_df[cols]
    
    # Sort by the new metric, ascending (lower is better)
    return results_df.sort_values('final_score', ascending=True)